### Visualizing the image right before prediction

In [ ]:
import random
from pathlib import Path

import torch
import matplotlib.pyplot as plt
from PIL import Image

from utils import get_dataset_roots, collect_images
from mlep_fast_evaluator import get_mlep_transform


def visualize_random_mlep_preprocessed_image(
    load_size=256,
    crop_size=224,
    no_resize=False,
    no_crop=True,
    seed=None,
):
    """
    Picks one random dataset, one random image, applies the exact MLEP preprocessing,
    and shows:
      1. original image
      2. preprocessed image after de-normalizing for visualization

    The actual tensor printed is the real tensor passed to the model.
    """

    if seed is not None:
        random.seed(seed)

    dataset_roots = get_dataset_roots()

    # Keep only datasets that exist and contain images
    valid_datasets = []
    for dataset_name, dataset_root in dataset_roots.items():
        dataset_root = Path(dataset_root)
        if not dataset_root.exists():
            continue

        samples = collect_images(dataset_root)
        if len(samples) > 0:
            valid_datasets.append((dataset_name, dataset_root, samples))

    if not valid_datasets:
        raise RuntimeError("No valid datasets found. Check your Datasets/ paths.")

    dataset_name, dataset_root, samples = random.choice(valid_datasets)
    sample = random.choice(samples)

    img_path = Path(sample["path"])
    original_img = Image.open(img_path).convert("RGB")

    transform = get_mlep_transform(
        load_size=load_size,
        crop_size=crop_size,
        no_resize=no_resize,
        no_crop=no_crop,
    )

    preprocessed_tensor = transform(original_img)          # [3, H, W]
    model_input_tensor = preprocessed_tensor.unsqueeze(0)  # [1, 3, H, W] -> this is passed to model

    # De-normalize only for visualization
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
    std = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

    raw_vis_tensor = preprocessed_tensor.cpu().clamp(0, 1)
    raw_vis_img = raw_vis_tensor.permute(1, 2, 0).numpy()

    denorm_tensor = preprocessed_tensor.cpu() * std + mean
    denorm_tensor = denorm_tensor.clamp(0, 1)
    denorm_img = denorm_tensor.permute(1, 2, 0).numpy()

    plt.figure(figsize=(15, 5))

    plt.subplot(1, 3, 1)
    plt.imshow(original_img)
    plt.title("Original image")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(raw_vis_img)
    plt.title("Raw normalized tensor\nclipped for display")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(denorm_img)
    plt.title("After preprocessing\n(de-normalized)")
    plt.axis("off")

    plt.tight_layout()
    plt.show()

    return {
        "dataset_name": dataset_name,
        "dataset_root": str(dataset_root),
        "image_id": sample["id"],
        "image_path": str(img_path),
        "true_label": sample["true_label"],
        "split": sample["split"],
        "model_input_tensor": model_input_tensor,
    }


sample_info = visualize_random_mlep_preprocessed_image()

### Finetuning and comparing results

In [ ]:
from utils import get_dataset_roots

get_dataset_roots()

In [ ]:
from finetune_mlep import finetune_mlep

result = finetune_mlep(
    dataset_name="gravex200k",
    weights_path="MLEP/pretrained/model_epoch_best.pth",
    max_per_class=5000,
    epochs=10,
    batch_size=32,
    lr=1e-5,
    num_workers=0,
    amp=False,
)

In [ ]:
from mlep_fast_evaluator import load_mlep_model, evaluate_all_datasets_mlep_fast

model, device = load_mlep_model(
    repo_dir="MLEP",
    weights_path=result["best_checkpoint"],
)

results = evaluate_all_datasets_mlep_fast(
    model=model,
    device=device,
    model_name="finetuned_MLEP_gravex200k_5k_per_class",
    batch_size=64,
    checkpoint_every=50,
    num_workers=0,
    max_batches=3,   # smoke test first
    threshold=0.5,
)

In [ ]:
results = evaluate_all_datasets_mlep_fast(
    model=model,
    device=device,
    model_name="finetuned_MLEP_gravex200k_5k_per_class_full",
    batch_size=64,
    checkpoint_every=50,
    num_workers=0,
    max_batches=None,
    threshold=0.5,
)

### 